# Exploratory Data Analysis: ULB Credit Card Transactions

Run this notebook from the cloned repository. It describes the actual CSV, its imbalance, amount/time patterns, and anonymized feature associations. It does not train a model or interpret V1-V28 as business variables.

**Dataset note:** `Time` is elapsed seconds from the first record, not a calendar timestamp. The file has no account ID or location columns, so account velocity and geolocation cannot be learned from it.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

candidates = [Path.cwd() / 'data/raw/creditcard.csv', Path.cwd().parent / 'data/raw/creditcard.csv']
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Put creditcard.csv in data/raw/ and open the repository folder in VS Code.')
df = pd.read_csv(DATA_PATH)
print('Using:', DATA_PATH.resolve())
print('Shape:', df.shape)
df.head()

## 1. Data quality and class balance

Inspect missing values and exact duplicates before modeling. The training script removes exact duplicate rows before splitting to reduce train/holdout overlap; keep that preprocessing decision documented because repeated events can sometimes be legitimate.

In [ ]:
print('Missing cells:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Column types:\n', df.dtypes.value_counts())
counts = df['Class'].value_counts().sort_index().rename(index={0:'Legitimate (0)', 1:'Fraud (1)'})
summary = pd.DataFrame({'transactions': counts, 'percent': counts / len(df) * 100})
summary

In [ ]:
ax = summary['transactions'].plot(kind='bar', color=['steelblue','crimson'], title='Strong class imbalance in the original CSV')
ax.set_ylabel('Transactions'); ax.tick_params(axis='x', rotation=0); plt.show()
print(f"Majority/minority ratio: {summary.iloc[0,0] / summary.iloc[1,0]:.1f}:1")

## 2. Amount behavior

Amounts are skewed, so compare medians and quantiles as well as means. A high or low amount alone does not establish fraud.

In [ ]:
amount_summary = df.groupby('Class')['Amount'].agg(['count','mean','median','std','min','max'])
amount_summary.index = amount_summary.index.map({0:'Legitimate', 1:'Fraud'})
amount_summary

In [ ]:
plot_df = df[['Amount','Class']].copy()
plot_df['Class'] = plot_df['Class'].map({0:'Legitimate',1:'Fraud'})
plot_df['log_amount'] = np.log1p(plot_df['Amount'])
sns.histplot(data=plot_df, x='log_amount', hue='Class', bins=60, stat='density', common_norm=False, element='step')
plt.title('Log(1 + Amount) distribution by class'); plt.xlabel('log(1 + amount)'); plt.show()

## 3. Elapsed-time behavior

Do not label this as time-of-day: the data's `Time` variable is elapsed seconds. Compare the fraud rate across elapsed-time bins rather than raw histogram counts, since the classes have very different sizes.

In [ ]:
time_df = df[['Time','Class']].copy()
time_df['time_bin'] = pd.cut(time_df['Time'], bins=20, include_lowest=True)
time_rates = time_df.groupby('time_bin', observed=False).agg(transactions=('Class','size'), fraud=('Class','sum'))
time_rates['fraud_rate_pct'] = 100 * time_rates['fraud'] / time_rates['transactions']
time_rates

In [ ]:
plt.figure(figsize=(12,4))
time_rates['fraud_rate_pct'].plot(marker='o')
plt.title('Fraud-label rate across elapsed-time bins')
plt.xlabel('Elapsed-time bin'); plt.ylabel('Fraud rate (%)'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 4. Anonymized V-features

Rank features by standardized mean difference as a descriptive screen. This ranking is not a causal explanation and cannot reveal the original real-world meaning of the transformed columns.

In [ ]:
v_cols = [f'V{i}' for i in range(1,29) if f'V{i}' in df.columns]
groups = df.groupby('Class')[v_cols]
means = groups.mean()
stds = df[v_cols].std()
smd = (means.loc[1] - means.loc[0]) / stds
feature_rank = smd.abs().sort_values(ascending=False).rename('absolute_standardized_mean_difference').to_frame()
feature_rank.head(12)

In [ ]:
top = feature_rank.head(4).index.tolist()
fig, axes = plt.subplots(2,2,figsize=(11,7))
for ax, col in zip(axes.flat, top):
    sns.boxplot(data=df, x='Class', y=col, showfliers=False, ax=ax)
    ax.set_title(col); ax.set_xlabel('Class (0=legitimate, 1=fraud)')
plt.tight_layout(); plt.show()

In [ ]:
corr = df[v_cols + ['Class']].corr(numeric_only=True)['Class'].drop('Class').sort_values(key=lambda x: x.abs(), ascending=False)
print('Largest absolute Pearson correlations with Class (descriptive only):')
display(corr.head(10).to_frame('correlation_with_Class'))

## EDA takeaways

Write your own findings after reviewing the outputs. Include the exact CSV row count, class counts, duplicate count, amount distribution contrast, and the holdout design used in the later modeling stage. Avoid claims that anomaly equals fraud or that anonymized components have a known business interpretation.